In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from scipy import signal
import mne
import json
import seaborn as sns
from collections import defaultdict
import os
import glob


In [ ]:
"""
Fixed interpolation that maintains proper channel order alignment with electrode positions.
"""
def interpolate_missing_electrodes(raw_or_epochs, positions_file='electrode_positions.json', 
                                 force_interpolate=None, verbose=True):
    """
    Interpolate missing electrodes using custom positions with proper channel ordering.
    
    Parameters
    ----------
    raw_or_epochs : mne.Raw or mne.Epochs
        Input EEG data
    positions_file : str
        Path to JSON file with electrode positions
    force_interpolate : list or None
        Channels to force re-interpolation
    verbose : bool
        Print progress
    
    Returns
    -------
    data_interpolated : mne.Raw or mne.Epochs
        Data with interpolated channels in correct order
    interpolated_channels : list
        List of interpolated channels
    """
    
    if force_interpolate is None:
        force_interpolate = []
    
    data_copy = raw_or_epochs.copy()
    
    # Load positions
    with open(positions_file, 'r') as f:
        electrode_data = json.load(f)
    
    ch_pos = {}
    for ch, pos_dict in electrode_data.items():
        ch_pos[ch] = np.array([pos_dict['x'], pos_dict['y'], pos_dict['z']])
    
    # Get the expected channel order from JSON (this is the target order)
    expected_channel_order = list(electrode_data.keys())
    
    # Rename FP1 -> Fp1 if needed
    if 'FP1' in data_copy.ch_names and 'FP1' not in ch_pos and 'Fp1' in ch_pos:
        data_copy.rename_channels({'FP1': 'Fp1'})
        if verbose:
            print("Renamed FP1 -> Fp1")
    
    # Find channels to interpolate
    current = set(data_copy.ch_names)
    available = set(ch_pos.keys())
    missing = list(available - current)
    forced = [ch for ch in force_interpolate if ch in current]
    to_interpolate = list(set(missing + forced))
    
    if not to_interpolate:
        # Even if no interpolation needed, ensure correct channel order
        try:
            # Only reorder channels that exist in both data and expected order
            existing_in_expected = [ch for ch in expected_channel_order if ch in data_copy.ch_names]
            data_copy = data_copy.reorder_channels(existing_in_expected)
            if verbose:
                print("Reordered existing channels to match electrode positions")
        except:
            if verbose:
                print("Could not reorder channels - some channels may not match electrode positions")
        return data_copy, []
    
    if verbose:
        print(f"Interpolating {len(to_interpolate)} channels: {to_interpolate}")
    
    # Source channels for interpolation
    sources = {ch: ch_pos[ch] for ch in data_copy.ch_names 
               if ch not in to_interpolate and ch in ch_pos}
    
    # Store original data before removing channels
    original_data = data_copy.get_data()
    original_ch_names = data_copy.ch_names.copy()
    ch_to_idx = {ch: idx for idx, ch in enumerate(original_ch_names)}
    
    # Remove existing channels that will be re-interpolated
    existing_to_remove = [ch for ch in to_interpolate if ch in data_copy.ch_names]
    if existing_to_remove:
        data_copy = data_copy.pick_channels([ch for ch in data_copy.ch_names if ch not in existing_to_remove])
    
    # Interpolate each channel
    interpolated_data = []
    successful = []
    
    for ch in to_interpolate:
        if ch not in ch_pos:
            continue
        
        # Find 6 nearest neighbors
        target_pos = ch_pos[ch]
        distances = [(name, np.linalg.norm(target_pos - pos)) for name, pos in sources.items()]
        neighbors = sorted(distances, key=lambda x: x[1])[:6]
        
        if not neighbors:
            if verbose:
                print(f"  {ch}: No neighbors found, skipping")
            continue
        
        # Weights by inverse squared distance
        weights = np.array([1/(d**2) if d > 1e-10 else 1.0 for _, d in neighbors])
        weights /= weights.sum()
        
        # Weighted average
        neighbor_idx = [ch_to_idx[name] for name, _ in neighbors if name in ch_to_idx]
        if not neighbor_idx:
            if verbose:
                print(f"  {ch}: No valid neighbor indices found, skipping")
            continue
            
        if original_data.ndim == 2:  # Raw
            signal = sum(w * original_data[idx] for w, idx in zip(weights[:len(neighbor_idx)], neighbor_idx))
        else:  # Epochs
            signal = sum(w * original_data[:, idx, :] for w, idx in zip(weights[:len(neighbor_idx)], neighbor_idx))
        
        interpolated_data.append(signal)
        successful.append(ch)
        
        if verbose:
            neighbor_names = [name for name, _ in neighbors[:len(neighbor_idx)]]
            print(f"  {ch}: from {neighbor_names}")
    
    # Add interpolated channels back
    if successful:
        if original_data.ndim == 2:
            interp_array = np.array(interpolated_data)
        else:
            interp_array = np.transpose(np.array(interpolated_data), (1, 0, 2))
        
        interp_info = mne.create_info(successful, data_copy.info['sfreq'], ['eeg'] * len(successful))
        
        if original_data.ndim == 2:
            interp_obj = mne.io.RawArray(interp_array, interp_info, verbose=False)
        else:
            interp_obj = mne.EpochsArray(interp_array, interp_info, verbose=False)
        
        data_copy = data_copy.add_channels([interp_obj], force_update_info=True)
    
    # CRITICAL FIX: Reorder channels to match electrode positions JSON
    try:
        # Only include channels that exist in the final data
        final_channel_order = [ch for ch in expected_channel_order if ch in data_copy.ch_names]
        
        if verbose:
            print(f"Original order: {data_copy.ch_names[:10]}...")
            print(f"Target order:   {final_channel_order[:10]}...")
        
        # Reorder to match electrode positions
        data_copy = data_copy.reorder_channels(final_channel_order)
        
        if verbose:
            print(f"✓ Reordered {len(data_copy.ch_names)} channels to match electrode positions")
            print(f"Final order:    {data_copy.ch_names[:10]}...")
            
    except Exception as e:
        if verbose:
            print(f"Warning: Could not reorder channels: {e}")
            print("Channel order may not match electrode positions!")
    
    if verbose:
        print(f"✓ Interpolated {len(successful)} channels")
        print(f"✓ Final data has {len(data_copy.ch_names)} channels in correct order")
    
    return data_copy, successful


def parse_file_metadata(file_path):
    parts = file_path.split('/')
    
    drug_condition = 'Psilocybin' if 'Psilocybin' in parts else 'Control'
    
    task = None
    for part in parts:
        if part in ['Meditation', 'Music', 'Resting state', 'Video']:
            task = part
            break
    
    filename = os.path.basename(file_path)
    parts = filename.split('_')
    
    # Extract subject number from parts like ['ctrl', 'sub001', 'meditation.set']
    subject_id = parts[0] if len(parts) > 0 else filename.split('.')[0]
    subject_number = None
    
    if len(parts) > 1 and 'sub' in parts[1]:
        subject_number = parts[1].replace('sub', '')
    
    is_trained = subject_number.startswith('2') if subject_number and subject_number[0].isdigit() else None
    
    return {
        'drug_condition': drug_condition,
        'task': task,
        'subject_id': subject_id,
        'subject_number': subject_number,
        'is_trained': is_trained,
        'file_path': file_path
    }

def main():
    base_dirs = [
        '/home/novicky/psiconnect/Control',
        '/home/novicky/psiconnect/Psilocybin'
    ]
    
    conditions = ['Meditation', 'Music', 'Resting state', 'Video']
    
    file_paths = []
    for base_dir in base_dirs:
        for condition in conditions:
            pattern = os.path.join(base_dir, condition, '*.set')
            files = glob.glob(pattern)
            file_paths.extend(files)
    
    file_paths.sort()
    print(f"Found {len(file_paths)} .set files")
    
    datasets = []
    file_metadata = []
    
    for i, file_path in enumerate(file_paths):
        print(f"Processing {i+1}/{len(file_paths)}: {os.path.basename(file_path)}")
        
        raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
        raw.pick_types(eeg=True, exclude='bads')
        
        raw_interp, interp_channels = interpolate_missing_electrodes(
            raw, 
            positions_file='/home/novicky/clean_code/electrode_positions.json',
            verbose=False
        )
        
        raw_interp.resample(250, npad='auto', verbose=False)
        
        data = raw_interp.get_data()
        datasets.append(data)
        
        metadata = parse_file_metadata(file_path)
        file_metadata.append(metadata)
    
    # Save datasets and metadata
    print("Saving datasets...")
    with open('datasets.pkl', 'wb') as f:
        pickle.dump(datasets, f)
    
    print("Saving metadata...")
    with open('metadata.pkl', 'wb') as f:
        pickle.dump(file_metadata, f)
    
    print(f"Saved {len(datasets)} datasets and metadata")

if __name__ == "__main__":
    main()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

def compute_temporal_statistics(correlations, positions, threshold=1.0, sfreq=250):
    above_threshold = correlations > threshold
    n_above = np.sum(above_threshold)
    
    below_threshold = correlations < -threshold
    n_below = np.sum(below_threshold)
    
    positions_above = positions[above_threshold]
    correlations_above = correlations[above_threshold]
    
    positions_below = positions[below_threshold]
    correlations_below = correlations[below_threshold]
    
    if len(positions_above) > 1:
        intervals_above = np.diff(positions_above)
        intervals_above_sec = intervals_above / sfreq
        mean_lag_above = np.mean(intervals_above_sec)
        std_lag_above = np.std(intervals_above_sec)
        median_lag_above = np.median(intervals_above_sec)
    else:
        intervals_above_sec = np.array([])
        mean_lag_above = np.nan
        std_lag_above = np.nan
        median_lag_above = np.nan
    
    if len(positions_below) > 1:
        intervals_below = np.diff(positions_below)
        intervals_below_sec = intervals_below / sfreq
        mean_lag_below = np.mean(intervals_below_sec)
        std_lag_below = np.std(intervals_below_sec)
        median_lag_below = np.median(intervals_below_sec)
    else:
        intervals_below_sec = np.array([])
        mean_lag_below = np.nan
        std_lag_below = np.nan
        median_lag_below = np.nan
    
    total_duration = (positions[-1] - positions[0]) / sfreq
    
    rate_above = n_above / total_duration if total_duration > 0 else 0
    rate_below = n_below / total_duration if total_duration > 0 else 0
    
    return {
        'n_above_threshold': n_above,
        'rate_above': rate_above,
        'mean_lag_above': mean_lag_above,
        'std_lag_above': std_lag_above,
        'median_lag_above': median_lag_above,
        'intervals_above': intervals_above_sec,
        'positions_above': positions_above,
        'correlations_above': correlations_above,
        
        'n_below_threshold': n_below,
        'rate_below': rate_below,
        'mean_lag_below': mean_lag_below,
        'std_lag_below': std_lag_below,
        'median_lag_below': median_lag_below,
        'intervals_below': intervals_below_sec,
        'positions_below': positions_below,
        'correlations_below': correlations_below,
        
        'threshold': threshold,
        'total_duration': total_duration
    }


def build_spatial_smoothing_matrix(electrode_positions, k=3.0):
    """
    Build spatial smoothing matrix for frame-by-frame correlation.
    
    Parameters:
    -----------
    electrode_positions : array (n_electrodes, 3)
        x, y, z coordinates
    k : float
        Decay rate (default: 3.0)
    
    Returns:
    --------
    W_smooth : array (n_electrodes, n_electrodes)
        Row-normalized smoothing matrix
    distances : array (n_electrodes, n_electrodes)
        Pairwise distances
    mean_distance : float
        Mean inter-electrode distance
    """
    n_electrodes = len(electrode_positions)
    
    # Compute pairwise distances
    distances = cdist(electrode_positions, electrode_positions, metric='euclidean')
    
    # Mean distance (excluding diagonal)
    mask = ~np.eye(n_electrodes, dtype=bool)
    mean_distance = np.mean(distances[mask])
    
    # Normalize distances
    W = distances / mean_distance
    
    # Compute spatial weights with exponential decay
    if k == 0:
        spatial_weights = W
    else:
        spatial_weights = np.exp(-k * W)
    
    # Row-normalize so each row sums to 1
    W_smooth = spatial_weights / spatial_weights.sum(axis=1, keepdims=True)
    
    return W_smooth, distances, mean_distance


def spatiotemporal_correlation(pattern, data_window, W_smooth):
    """
    Compute spatiotemporal correlation using frame-by-frame spatial correlation.
    
    For each timepoint:
    1. Spatially smooth both pattern and data frames
    2. Compute correlation between smoothed frames
    3. Average correlation across all timepoints
    
    This captures:
    - Spatial structure at each moment (via smoothing)
    - Temporal evolution (via frame-by-frame comparison)
    - Propagation (spatial pattern moving through time)
    
    Parameters:
    -----------
    pattern : array (n_electrodes, n_timepoints)
        Template pattern showing spatiotemporal propagation
    data_window : array (n_electrodes, n_timepoints)
        Data window to compare
    W_smooth : array (n_electrodes, n_electrodes)
        Spatial smoothing matrix
    
    Returns:
    --------
    correlation : float
        Spatiotemporal correlation in [-1, 1]
    """
    n_electrodes, n_timepoints = pattern.shape
    
    frame_correlations = []
    
    # For each timepoint, compute spatial correlation
    for t in range(n_timepoints):
        # Get spatial frame at time t
        pattern_frame = pattern[:, t]  # (n_electrodes,)
        data_frame = data_window[:, t]  # (n_electrodes,)
        
        # Spatially smooth both frames
        # This allows nearby electrodes to contribute to each location
        pattern_smooth = W_smooth @ pattern_frame
        data_smooth = W_smooth @ data_frame
        
        # Compute correlation between smoothed spatial patterns
        if np.std(pattern_smooth) > 1e-6 and np.std(data_smooth) > 1e-6:
            corr = np.corrcoef(pattern_smooth, data_smooth)[0, 1]
            frame_correlations.append(corr)
        else:
            # If either frame is constant, correlation is undefined
            frame_correlations.append(0.0)
    
    # Average correlation across time
    # High value means spatial structure matches at each timepoint
    if len(frame_correlations) > 0:
        avg_correlation = np.mean(frame_correlations)
    else:
        avg_correlation = 0.0
    
    # Clip to valid range
    avg_correlation = np.clip(avg_correlation, -1.0, 1.0)
    
    return avg_correlation


def run_analysis_for_k(k_value):
    """
    Run the full analysis pipeline for a specific k value.
    NO correlation threshold applied - all correlations are saved.
    
    Parameters:
    -----------
    k_value : float
        Spatial decay parameter for smoothing
    
    Returns:
    --------
    results : dict
        Complete results dictionary
    """
    
    print(f"\n{'='*80}")
    print(f"RUNNING ANALYSIS WITH k={k_value} (NO CORRELATION THRESHOLD)")
    print(f"{'='*80}")
    
    # Load pre-saved data
    print("Loading datasets...")
    with open('/home/novicky/datasets.pkl', 'rb') as f:
        datasets = pickle.load(f)

    print("Loading metadata...")
    with open('/home/novicky/metadata.pkl', 'rb') as f:
        file_metadata = pickle.load(f)

    print("Loading cluster averages...")
    with open('/home/novicky/clean_code/similarity_blocks_with_polarity_3/cluster_averages.pkl', 'rb') as f:
        cluster_averages = pickle.load(f)

    print("Loading or creating electrode positions...")
    # REPLACE THIS with your actual electrode positions!
    # For now, using 8x8 grid as placeholder
    n_electrodes = 64
    grid_size = 8
    spacing = 10.0  # mm
    x = np.repeat(np.arange(grid_size) * spacing, grid_size)
    y = np.tile(np.arange(grid_size) * spacing, grid_size)
    z = np.zeros(n_electrodes)
    electrode_positions = np.column_stack([x, y, z])

    print(f"\nLoaded {len(datasets)} datasets")
    print(f"Loaded {len(cluster_averages)} clusters")

    # Build spatial smoothing matrix with specified k
    print(f"\nBuilding spatial smoothing matrix with k={k_value}...")
    W_smooth, distances, mean_distance = build_spatial_smoothing_matrix(
        electrode_positions, k=k_value
    )

    print(f"Mean electrode distance: {mean_distance:.2f} mm")
    print(f"Smoothing matrix built: {W_smooth.shape}")

    # Use the actual number of clusters available
    top_clusters = len(cluster_averages)
    print(f"\nAnalyzing {top_clusters} clusters with spatiotemporal correlation")
    print("NOTE: Saving ALL correlations (no threshold)")

    # Track cluster-specific statistics
    cluster_stats = {}
    for cluster_id in range(top_clusters):
        cluster_stats[cluster_id] = {
            'drug': {'Control': [], 'Psilocybin': []},
            'task': {'Meditation': [], 'Music': [], 'Resting state': [], 'Video': []},
            'training': {'Trained': [], 'Non-trained': []}
        }

    # Store ALL sliding window correlations (dataset_idx, times, polarities, correlations)
    cluster_activation_times = {}
    for cluster_id in range(top_clusters):
        cluster_activation_times[cluster_id] = []

    # Co-activation and lead-lag matrices (these will use positive correlations for compatibility)
    co_activation_matrix = np.zeros((top_clusters, top_clusters))
    lead_lag_matrix = np.zeros((top_clusters, top_clusters))
    lead_lag_counts = np.zeros((top_clusters, top_clusters))

    # Store all intervals for each cluster
    cluster_intervals = {}
    for cluster_id in range(top_clusters):
        cluster_intervals[cluster_id] = []

    print(f"\nAnalyzing datasets against clusters (k={k_value}, saving ALL correlations)...")
    for dataset_idx, (data, metadata) in enumerate(zip(datasets, file_metadata)):
        if dataset_idx % 10 == 0:
            print(f"  Processing dataset {dataset_idx}/{len(datasets)}...")
        
        # Store all activations for this dataset for co-activation analysis
        dataset_activations = []  # List of (cluster_id, timepoint_in_seconds)
        
        # Correlate dataset with each cluster individually
        for cluster_id in range(top_clusters):
            cluster_pattern = cluster_averages[cluster_id]  # Shape: (64, time_points)
            
            # Slide cluster pattern across dataset timeline
            dataset_channels, dataset_timepoints = data.shape
            cluster_channels, cluster_timepoints = cluster_pattern.shape
            
            if dataset_channels != cluster_channels:
                continue  # Skip if channel mismatch
            
            correlations = []
            timepoints = []
            
            # Slide window across dataset
            for t in range(0, dataset_timepoints - cluster_timepoints + 1, cluster_timepoints):
                dataset_window = data[:, t:t + cluster_timepoints]
                
                # SPATIOTEMPORAL CORRELATION
                corr = spatiotemporal_correlation(
                    cluster_pattern, 
                    dataset_window, 
                    W_smooth
                )
                
                correlations.append(corr)
                timepoints.append(t)
            
            # Convert to arrays
            correlations = np.array(correlations)
            timepoints = np.array(timepoints)
            times_sec = timepoints / 250.0
            
            # Store ALL correlations (no thresholding!)
            # Assign polarity based on sign
            polarities = np.sign(correlations)
            
            if len(times_sec) > 0:
                # Store ALL activation times with their correlations
                cluster_activation_times[cluster_id].append((
                    dataset_idx, 
                    times_sec, 
                    polarities,
                    correlations
                ))
                
                # For co-activation analysis, only use positive correlations > 0.7 for compatibility
                positive_mask = correlations > 0.7
                positive_times_sec = times_sec[positive_mask]
                for time_sec in positive_times_sec:
                    dataset_activations.append((cluster_id, time_sec))
                
                # Calculate intervals between ALL timepoints
                if len(times_sec) > 1:
                    intervals = np.diff(times_sec)
                    cluster_intervals[cluster_id].extend(intervals)
            
            # Count positive activations for statistics (backward compatibility)
            positive_mask = correlations > 0.7
            activations = np.sum(positive_mask)
            
            # Normalize by dataset duration (activations per minute)
            dataset_duration_minutes = dataset_timepoints / (250.0 * 60.0)  # 250 Hz, convert to minutes
            normalized_activations = activations / dataset_duration_minutes if dataset_duration_minutes > 0 else 0
            
            # Store cluster-specific activations by condition (normalized)
            cluster_stats[cluster_id]['drug'][metadata['drug_condition']].append(normalized_activations)
            if metadata['task']:
                cluster_stats[cluster_id]['task'][metadata['task']].append(normalized_activations)
            if metadata['is_trained'] is not None:
                training_key = 'Trained' if metadata['is_trained'] else 'Non-trained'
                cluster_stats[cluster_id]['training'][training_key].append(normalized_activations)
        
        # Analyze co-activations and lead-lag relationships for this dataset
        if len(dataset_activations) > 1:
            # Sort by time
            dataset_activations.sort(key=lambda x: x[1])
            
            # Co-activation analysis (50ms window)
            time_window = 0.05  # 50ms
            for i, (cluster_a, time_a) in enumerate(dataset_activations):
                for j, (cluster_b, time_b) in enumerate(dataset_activations):
                    if i != j and cluster_a != cluster_b:
                        if abs(time_a - time_b) <= time_window:
                            co_activation_matrix[cluster_a, cluster_b] += 1
            
            # Lead-lag analysis
            for i, (cluster_a, time_a) in enumerate(dataset_activations):
                for j, (cluster_b, time_b) in enumerate(dataset_activations):
                    if cluster_a != cluster_b:
                        time_diff = time_b - time_a
                        # Only consider reasonable lead-lag relationships (within 5 seconds)
                        if 0 < time_diff <= 5.0:
                            lead_lag_matrix[cluster_a, cluster_b] += time_diff
                            lead_lag_counts[cluster_a, cluster_b] += 1

    print("Computing statistics...")

    # Post-processing: Calculate average lead-lag times
    average_lead_lag_matrix = np.zeros((top_clusters, top_clusters))
    for i in range(top_clusters):
        for j in range(top_clusters):
            if lead_lag_counts[i, j] > 0:
                average_lead_lag_matrix[i, j] = lead_lag_matrix[i, j] / lead_lag_counts[i, j]

    # Normalize co-activation matrix by total activations
    total_activations_per_cluster = np.zeros(top_clusters)
    for cluster_id in range(top_clusters):
        for _, times, _, _ in cluster_activation_times[cluster_id]:
            total_activations_per_cluster[cluster_id] += len(times)

    co_activation_probability = np.zeros((top_clusters, top_clusters))
    for i in range(top_clusters):
        for j in range(top_clusters):
            if i != j and total_activations_per_cluster[i] > 0:
                co_activation_probability[i, j] = co_activation_matrix[i, j] / total_activations_per_cluster[i]

    # Analyze polarity switching
    print("Analyzing polarity switches...")
    polarity_switch_stats = []

    for cluster_id in range(top_clusters):
        all_switches = []
        total_positive = 0
        total_negative = 0
        
        for dataset_idx, times, polarities, correlations in cluster_activation_times[cluster_id]:
            # Count positive and negative correlations
            total_positive += np.sum(polarities > 0)
            total_negative += np.sum(polarities < 0)
            
            # Find polarity switches
            if len(polarities) > 1:
                polarity_diffs = np.diff(polarities)
                switch_indices = np.where(polarity_diffs != 0)[0]
                
                if len(switch_indices) > 0:
                    # Calculate intervals between switches
                    switch_times = times[switch_indices]
                    if len(switch_times) > 1:
                        switch_intervals = np.diff(switch_times)
                        all_switches.extend(switch_intervals)
        
        # Compute statistics for this cluster
        if len(all_switches) > 0:
            all_switches = np.array(all_switches)
            mean_interval = np.mean(all_switches)
            std_interval = np.std(all_switches)
            cv = std_interval / mean_interval if mean_interval > 0 else np.inf
            
            polarity_switch_stats.append({
                'cluster_id': cluster_id,
                'n_switches': len(all_switches),
                'mean_interval': mean_interval,
                'std_interval': std_interval,
                'cv': cv,
                'total_positive': total_positive,
                'total_negative': total_negative,
                'polarity_ratio': total_positive / total_negative if total_negative > 0 else np.inf
            })

    # Sort by number of switches
    polarity_switch_stats_sorted = sorted(polarity_switch_stats, key=lambda x: x['n_switches'], reverse=True)

    # Find clusters with regular polarity switching (low CV, many switches)
    regular_switchers = sorted([s for s in polarity_switch_stats if s['cv'] < 1.0 and s['n_switches'] >= 10], 
                              key=lambda x: x['cv'])[:10]

    # Create heatmap matrices for visualization
    clusters = list(range(top_clusters))

    # Drug condition matrix (Control vs Psilocybin)
    drug_matrix = np.zeros((top_clusters, 2))
    drug_labels = ['Control', 'Psilocybin']

    # Task condition matrix
    task_matrix = np.zeros((top_clusters, 4))
    task_labels = ['Meditation', 'Music', 'Resting state', 'Video']

    # Training matrix
    training_matrix = np.zeros((top_clusters, 2))
    training_labels = ['Non-trained', 'Trained']

    # Fill matrices with mean activation rates
    for cluster_id in range(top_clusters):
        # Drug conditions
        for i, drug in enumerate(['Control', 'Psilocybin']):
            activations = cluster_stats[cluster_id]['drug'][drug]
            if activations:
                drug_matrix[cluster_id, i] = np.mean(activations)
        
        # Task conditions
        for i, task in enumerate(['Meditation', 'Music', 'Resting state', 'Video']):
            activations = cluster_stats[cluster_id]['task'][task]
            if activations:
                task_matrix[cluster_id, i] = np.mean(activations)
        
        # Training conditions
        for i, training in enumerate(['Non-trained', 'Trained']):
            activations = cluster_stats[cluster_id]['training'][training]
            if activations:
                training_matrix[cluster_id, i] = np.mean(activations)

    # Calculate all statistics

    # Find clusters with strongest drug preferences
    drug_ratios = []
    for cluster_id in range(top_clusters):
        control_rate = drug_matrix[cluster_id, 0]
        psilocybin_rate = drug_matrix[cluster_id, 1]
        if control_rate > 0 and psilocybin_rate > 0:
            ratio = psilocybin_rate / control_rate
            drug_ratios.append((cluster_id, ratio, psilocybin_rate, control_rate))

    # Sort by strongest psilocybin preference
    psilocybin_preferred = sorted([x for x in drug_ratios if x[1] > 1.2], key=lambda x: x[1], reverse=True)[:10]
    control_preferred = sorted([x for x in drug_ratios if x[1] < 0.8], key=lambda x: x[1])[:10]

    # Find most active clusters overall
    overall_activity = []
    for cluster_id in range(top_clusters):
        mean_activity = np.mean([drug_matrix[cluster_id, 0], drug_matrix[cluster_id, 1]])
        overall_activity.append((cluster_id, mean_activity))

    most_active = sorted(overall_activity, key=lambda x: x[1], reverse=True)[:min(15, top_clusters)]

    # Interval statistics
    interval_stats = []
    for cluster_id in range(top_clusters):
        if len(cluster_intervals[cluster_id]) > 2:
            intervals = np.array(cluster_intervals[cluster_id])
            mean_interval = np.mean(intervals)
            std_interval = np.std(intervals)
            cv = std_interval / mean_interval if mean_interval > 0 else np.inf
            interval_stats.append((cluster_id, mean_interval, std_interval, cv, len(intervals)))

    regular_clusters = sorted([x for x in interval_stats if x[3] < 1.0], key=lambda x: x[3])[:10]

    # Top co-activating cluster pairs
    co_activation_pairs = []
    for i in range(top_clusters):
        for j in range(i+1, top_clusters):
            prob = (co_activation_probability[i, j] + co_activation_probability[j, i]) / 2
            if prob > 0.01:  # At least 1% co-activation
                co_activation_pairs.append((i, j, prob, co_activation_matrix[i, j] + co_activation_matrix[j, i]))

    top_coactivations = sorted(co_activation_pairs, key=lambda x: x[2], reverse=True)[:10]

    # Top lead-lag relationships
    lead_lag_relationships = []
    for i in range(top_clusters):
        for j in range(top_clusters):
            if i != j and lead_lag_counts[i, j] >= 5:  # At least 5 observations
                avg_lag = average_lead_lag_matrix[i, j]
                count = lead_lag_counts[i, j]
                lead_lag_relationships.append((i, j, avg_lag, count))

    strong_relationships = sorted(lead_lag_relationships, key=lambda x: x[3], reverse=True)[:10]

    # Save all results
    results = {
        'k_value': k_value,
        'correlation_threshold': None,  # No threshold used!
        'cluster_stats': cluster_stats,
        'drug_matrix': drug_matrix,
        'task_matrix': task_matrix,
        'training_matrix': training_matrix,
        'drug_labels': drug_labels,
        'task_labels': task_labels,
        'training_labels': training_labels,
        'psilocybin_preferred': psilocybin_preferred,
        'control_preferred': control_preferred,
        'most_active': most_active,
        'drug_ratios': drug_ratios,
        'overall_activity': overall_activity,
        # Temporal analysis results
        'cluster_intervals': cluster_intervals,
        'cluster_activation_times': cluster_activation_times,
        'co_activation_matrix': co_activation_matrix,
        'co_activation_probability': co_activation_probability,
        'lead_lag_matrix': lead_lag_matrix,
        'average_lead_lag_matrix': average_lead_lag_matrix,
        'lead_lag_counts': lead_lag_counts,
        'interval_stats': interval_stats,
        'regular_clusters': regular_clusters,
        'top_coactivations': top_coactivations,
        'strong_relationships': strong_relationships,
        # Polarity switching results
        'polarity_switch_stats': polarity_switch_stats,
        'regular_switchers': regular_switchers,
        # Spatial info
        'W_smooth': W_smooth,
        'electrode_positions': electrode_positions,
        'mean_distance': mean_distance,
        'spatial_k': k_value
    }
    
    print(f"\n✓ Analysis complete for k={k_value}")
    
    return results


# ============================================================================
# MAIN EXECUTION: Test different k values (NO THRESHOLD)
# ============================================================================

if __name__ == "__main__":
    
    print("="*80)
    print("PARAMETER SWEEP: TESTING DIFFERENT k VALUES")
    print("NO CORRELATION THRESHOLD - ALL CORRELATIONS SAVED")
    print("="*80)
    
    # Define k values to test
    k_values = [0]#1.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0]
    
    print(f"\nTesting {len(k_values)} k values: {k_values}")
    
    # Store all results
    all_results = {}
    
    # Run analysis for each k value
    for k in k_values:
        config_name = f"k{k}_no_threshold"
        print(f"\n{'='*80}")
        print(f"Configuration: {config_name}")
        print(f"{'='*80}")
        
        results = run_analysis_for_k(k_value=k)
        all_results[config_name] = results
        
        # Save intermediate results
        output_file = f'/home/novicky/clean_code/similarity_blocks_with_polarity_3/cluster_analysis_{config_name}.pkl'
        with open(output_file, 'wb') as f:
            pickle.dump(results, f)
        print(f"✓ Saved to: {output_file}")
    
    # Save all results together
    print(f"\n{'='*80}")
    print("SAVING COMBINED RESULTS")
    print(f"{'='*80}")
    
    with open('/home/novicky/clean_code/similarity_blocks_with_polarity_3/cluster_analysis_all_k_values.pkl', 'wb') as f:
        pickle.dump(all_results, f)
    
    print("✓ All configurations saved to 'cluster_analysis_all_k_values.pkl'")
    print(f"\n{'='*80}")
    print("PARAMETER SWEEP COMPLETE")
    print(f"{'='*80}")